In [14]:
# %% [1] Загрузка и константы
from pathlib import Path
import numpy as np
import pandas as pd

In [15]:
PROC = Path("data/processed")
SNAPSHOT = pd.Timestamp("2018-09-01")   # день после конца периода (как в sql/06_rfm.sql)
P_START, P_END = pd.Timestamp("2017-01-01"), pd.Timestamp("2018-08-01")  # месяцы периода

items = pd.read_csv(PROC / "order_items.csv", parse_dates=["order_purchase_timestamp"])
orders = pd.read_csv(PROC / "orders.csv", parse_dates=["order_purchase_timestamp", "purchase_month"])

# Заказы delivered в периоде. Суммы считаем в центах (int), чтобы не было ошибок float
# при ранжировании одинаковых сумм (в SQL numeric точный).
d = items[(items.order_status == "delivered") & items.in_analysis_period].copy()
d["price_c"] = (d.price * 100).round().astype("int64")
ord_ = (d.groupby(["order_id", "customer_unique_id", "order_purchase_timestamp"], as_index=False)
          .agg(order_value_c=("price_c", "sum")))

assert ord_.order_id.nunique() == 96_211, "число заказов не совпало с SQL"
assert ord_.order_value_c.sum() == 1_318_102_713, "выручка периода не совпала с SQL (13 181 027.13)"
print("OK: 96 211 заказов, выручка 13 181 027.13")



OK: 96 211 заказов, выручка 13 181 027.13


In [16]:
# %% [2] RFM (повторяет sql/06_rfm.sql)
cust = (ord_.groupby("customer_unique_id")
        .agg(first_ts=("order_purchase_timestamp", "min"),
             last_ts=("order_purchase_timestamp", "max"),
             frequency=("order_id", "nunique"),
             monetary_c=("order_value_c", "sum"))
        .reset_index())
cust["monetary"] = cust.monetary_c / 100
cust["recency_days"] = (SNAPSHOT - cust.last_ts.dt.normalize()).dt.days

n = len(cust)
# CUME_DIST из SQL = rank(method="max") / N. Порядок операций 5 * (rank / n) как в SQL.
cust["r_score"] = np.ceil(5 * (cust.recency_days.rank(method="max", ascending=False) / n)).astype(int)
cust["m_score"] = np.ceil(5 * (cust.monetary_c.rank(method="max") / n)).astype(int)
cust["f_score"] = cust.frequency.clip(upper=3)

conds = [cust.f_score >= 2,
         (cust.r_score >= 4) & (cust.m_score >= 4),
         cust.r_score >= 4,
         cust.r_score == 3,
         cust.m_score >= 4]
names = ["1. Repeat buyers", "2. New high-value", "3. New low-value",
         "4. Cooling (1 order)", "5. Dormant high-value"]
cust["segment"] = np.select(conds, names, default="6. Lost low-value")

summary = (cust.groupby("segment")
           .agg(clients=("customer_unique_id", "size"),
                revenue_c=("monetary_c", "sum"),
                avg_monetary=("monetary", "mean"),
                avg_recency=("recency_days", "mean"),
                avg_frequency=("frequency", "mean")))
summary["revenue"] = summary.revenue_c / 100
summary["clients_pct"] = 100 * summary.clients / summary.clients.sum()
summary["revenue_pct"] = 100 * summary.revenue / summary.revenue.sum()
print(summary.drop(columns="revenue_c").round(2))

# Сверка с выводом sql/06_rfm.sql (сравниваем по номеру сегмента: имя 5-го сегмента изменено)
EXPECTED_SQL = {"1": (2789, 725355.04), "2": (14611, 3880648.82), "3": (21511, 1186856.94),
                "4": (18048, 2346286.40), "5": (13912, 3813420.04), "6": (22233, 1228459.89)}
for seg, row in summary.iterrows():
    c, r = EXPECTED_SQL[seg[0]]
    assert row.clients == c and abs(row.revenue - r) < 0.01, f"RFM расходится с SQL: {seg}"
print("OK: RFM-сегменты совпали с SQL")

# Пороги скоров: что значит «recent» и «high-value» (для README)
print("\nRecency по r_score:\n", cust.groupby("r_score").recency_days.agg(["min", "max", "size"]))
print("\nMonetary по m_score:\n", cust.groupby("m_score").monetary.agg(["min", "max", "size"]).round(2))

# Repeat Purchase Rate: классический и без разбитых корзин (аналог SQL 5e)
rpr = (cust.frequency >= 2).mean() * 100
rpr_24h = ((cust.last_ts - cust.first_ts) >= pd.Timedelta(days=1)).mean() * 100
print(f"\nRepeat rate классический: {rpr:.2f}% | после 24 часов: {rpr_24h:.2f}%")


                       clients  avg_monetary  avg_recency  avg_frequency  \
segment                                                                    
1. Repeat buyers          2789        260.08       222.11           2.11   
2. New high-value        14611        265.60        94.22           1.00   
3. New low-value         21511         55.17        92.01           1.00   
4. Cooling (1 order)     18048        130.00       222.30           1.00   
5. Dormant high-value    13912        274.11       394.84           1.00   
6. Lost low-value        22233         55.25       395.52           1.00   

                          revenue  clients_pct  revenue_pct  
segment                                                      
1. Repeat buyers        725355.04         3.00         5.50  
2. New high-value      3880648.82        15.69        29.44  
3. New low-value       1186856.94        23.10         9.00  
4. Cooling (1 order)   2346286.40        19.38        17.80  
5. Dormant high-val

In [18]:
# %% [3] Когортный retention
# Первый месяц клиента считаем по ВСЕМ delivered-заказам (включая 2016), как в sql/08.
dall = orders[orders.is_delivered]
first_m = dall.groupby("customer_unique_id").purchase_month.min().rename("cohort_month")
act = (dall[["customer_unique_id", "purchase_month"]].drop_duplicates()
       .merge(first_m, on="customer_unique_id"))
act = act[(act.purchase_month >= P_START) & (act.purchase_month <= P_END)].copy()
act["k"] = ((act.purchase_month.dt.year - act.cohort_month.dt.year) * 12
            + act.purchase_month.dt.month - act.cohort_month.dt.month)

# Сверка с sql/08: клиенто-месяцы активности, новые и «возвратные»
assert len(act) == 94_930, "активные клиенто-месяцы не совпали с SQL 08"
assert (act.k == 0).sum() == 93_094, "new_clients не совпали с SQL 08"
assert (act.k > 0).sum() == 1_836, "returning_clients не совпали с SQL 08"
print("\nOK: клиенто-месяцы совпали с SQL 08 (94 930 = 93 094 новых + 1 836 возвратных)")

# Матрица: когорта (месяц первого заказа, только период) x k (месяцев с первого заказа)
coh = act[act.cohort_month >= P_START]

# Матрица: когорта (строки) × k (столбцы), значения — число активных клиентов
mat = (coh
       .pivot_table(index="cohort_month", columns="k",
                    values="customer_unique_id",
                    aggfunc="count", fill_value=0))
# Гарантируем все k от 0 до 19: если на хвостовом k никто не вернулся — это 0%, а не пропуск
mat = mat.reindex(columns=range(0, 20), fill_value=0)

size = mat[0]   # k=0 — это и есть размер когорты
assert size.iloc[:3].tolist() == [717, 1628, 2503], \
    "размеры первых когорт не совпали с SQL 08"
# Маска ячеек, которых ещё не наблюдали (иначе 0 в хвосте выглядит как «никто не вернулся»)
max_k = pd.Series((P_END.year - mat.index.year) * 12 + (P_END.month - mat.index.month),
                  index=mat.index)
mask = mat.columns.to_numpy()[None, :] > max_k.to_numpy()[:, None]
retention = (mat.div(size, axis=0) * 100).mask(mask)

# Средний retention по k: суммируем только когорты, у которых этот k наблюдался
obs = ~mask
avg_ret = ((mat * obs).sum() / pd.Series((obs * size.to_numpy()[:, None]).sum(axis=0),
                                         index=mat.columns) * 100)
print("\nRetention, % (M1..M6, первые 6 когорт):")
print(retention.loc[:, 1:6].head(6).round(2))
print("\nСредневзвешенный retention по месяцам (M1..M12), %:")
print(avg_ret.loc[1:12].round(2).to_string())



OK: клиенто-месяцы совпали с SQL 08 (94 930 = 93 094 новых + 1 836 возвратных)

Retention, % (M1..M6, первые 6 когорт):
k                1     2     3     4     5     6
cohort_month                                    
2017-01-01    0.28  0.28  0.14  0.42  0.14  0.42
2017-02-01    0.18  0.31  0.12  0.43  0.12  0.25
2017-03-01    0.44  0.36  0.40  0.36  0.16  0.16
2017-04-01    0.62  0.22  0.18  0.27  0.27  0.35
2017-05-01    0.46  0.46  0.29  0.29  0.32  0.41
2017-06-01    0.49  0.40  0.43  0.30  0.40  0.36

Средневзвешенный retention по месяцам (M1..M12), %:
k
1     0.48
2     0.34
3     0.26
4     0.26
5     0.23
6     0.23
7     0.21
8     0.20
9     0.16
10    0.25
11    0.22
12    0.17


In [19]:
# %% [4] Сохранение для этапа 6 (экспорт в Tableau)
cust.drop(columns=["first_ts", "last_ts", "monetary_c"]).to_csv(PROC / "rfm.csv", index=False)

ret_long = mat.stack().rename("retained_clients").reset_index()
ret_long["cohort_size"] = ret_long.cohort_month.map(size)
ret_long = ret_long[ret_long.k <= ret_long.cohort_month.map(max_k)].copy()   # только наблюдаемые ячейки
ret_long["retention_pct"] = (100 * ret_long.retained_clients / ret_long.cohort_size).round(3)
ret_long.to_csv(PROC / "cohort_retention.csv", index=False)
print(f"\nСохранено: rfm.csv ({len(cust)} строк), cohort_retention.csv ({len(ret_long)} строк)")


Сохранено: rfm.csv (93104 строк), cohort_retention.csv (210 строк)


In [20]:
assert len(ret_long) == 210, f"ожидалось 210 наблюдаемых ячеек, получено {len(ret_long)}"

print("\n=== КЛЮЧЕВЫЕ ЦИФРЫ ===")
print(f"Repeat rate классический: {rpr:.2f}% | после 24 часов: {rpr_24h:.2f}%")
print("Recency по r_score:\n", cust.groupby("r_score").recency_days.agg(["min", "max", "size"]).to_string())
print("Monetary по m_score:\n", cust.groupby("m_score").monetary.agg(["min", "max", "size"]).round(2).to_string())
print("Средний retention M1..M12, %:\n", avg_ret.loc[1:12].round(2).to_string())
print("Размеры когорт:\n", size.to_string())


=== КЛЮЧЕВЫЕ ЦИФРЫ ===
Repeat rate классический: 3.00% | после 24 часов: 2.13%
Recency по r_score:
          min  max   size
r_score                 
1        384  604  18520
2        271  383  18615
3        180  270  18650
4         95  179  18664
5          3   94  18655
Monetary по m_score:
             min       max   size
m_score                         
1          0.85     39.89  18409
2         39.90     69.89  18798
3         69.90    109.89  18225
4        109.90    179.89  18944
5        179.90  13440.00  18728
Средний retention M1..M12, %:
 k
1     0.48
2     0.34
3     0.26
4     0.26
5     0.23
6     0.23
7     0.21
8     0.20
9     0.16
10    0.25
11    0.22
12    0.17
Размеры когорт:
 cohort_month
2017-01-01     717
2017-02-01    1628
2017-03-01    2503
2017-04-01    2256
2017-05-01    3451
2017-06-01    3037
2017-07-01    3752
2017-08-01    4057
2017-09-01    4004
2017-10-01    4328
2017-11-01    7060
2017-12-01    5338
2018-01-01    6842
2018-02-01    6288
2018-03-01